# Stage 2: PlantVillage + Mendeley Tomato ResNet-50 training

Run this notebook in Google Colab with **Runtime -> Change runtime type -> T4 GPU**. It reads the regenerated Stage 1 manifest and summary for the merged 55,914-image dataset (including the 1,609 Mendeley field-condition tomato images). Class weights are calculated from the current summary.json training counts; validation and test data are never augmented.


In [ ]:
# Set this to the project directory containing the updated Stage 1 data in Colab.
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path('/content/plant-disease-assistant')  # Edit if your Colab path differs.
AI_SERVICE = PROJECT_ROOT / 'ai-service'
DATA_ROOT = AI_SERVICE / 'data' / 'plantvillage'
IMAGE_DIR = DATA_ROOT / 'color'  # Explicitly use the merged PlantVillage + Mendeley image tree.
OUTPUT_DIR = AI_SERVICE / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(AI_SERVICE))

assert IMAGE_DIR.is_dir(), f'Missing merged Stage 1 color dataset at {IMAGE_DIR}'
with (DATA_ROOT / 'metadata' / 'summary.json').open(encoding='utf-8') as stream:
    stage1_summary = json.load(stream)
assert stage1_summary['total_images'] == 55914, (
    f"Expected updated 55,914-image dataset; found {stage1_summary['total_images']}. "
    'Upload/sync the regenerated Stage 1 dataset before training.'
)
assert stage1_summary['image_directory'] == 'color'
print('Using merged dataset:', IMAGE_DIR)
print('Images:', stage1_summary['total_images'], '| classes:', stage1_summary['class_count'])


In [ ]:
import csv
import torch
from PIL import Image
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet50_Weights, resnet50

from preprocess_images import ImagePreprocessor

SEED = 42
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 12
PATIENCE = 3

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# Read exactly the regenerated Stage 1 label mapping, splits, and class distribution.
from collections import Counter

metadata_dir = DATA_ROOT / 'metadata'
with (metadata_dir / 'class_to_index.json').open(encoding='utf-8') as stream:
    class_to_index = json.load(stream)
with (metadata_dir / 'summary.json').open(encoding='utf-8') as stream:
    summary = json.load(stream)
with (metadata_dir / 'splits.csv').open(newline='', encoding='utf-8') as stream:
    records = list(csv.DictReader(stream))

index_to_class = [name for name, _ in sorted(class_to_index.items(), key=lambda item: item[1])]
assert summary['total_images'] == len(records) == 55914
assert summary['class_count'] == len(index_to_class) == 38
assert Counter(row['split'] for row in records) == Counter(summary['split_sizes'])
train_counts = torch.tensor(
    [summary['class_distribution'][class_name]['train'] for class_name in index_to_class],
    dtype=torch.float32,
)
assert int(train_counts.sum()) == summary['split_sizes']['train']
# Recompute balanced weights from the NEW training distribution: N / (C * n_class).
class_weights = train_counts.sum() / (len(index_to_class) * train_counts)
print('Classes:', len(index_to_class), '| train samples:', int(train_counts.sum()))
print('Train-count range:', int(train_counts.min()), 'to', int(train_counts.max()))
print('Class-weight range:', float(class_weights.min()), 'to', float(class_weights.max()))
print('Recomputed class weights:')
for class_name, count, weight in zip(index_to_class, train_counts.tolist(), class_weights.tolist()):
    print(f'  {class_name}: n={int(count)}, weight={weight:.6f}')


In [ ]:
class PlantVillageManifestDataset(Dataset):
    def __init__(self, split, transform):
        self.image_dir = IMAGE_DIR
        self.records = [row for row in records if row['split'] == split]
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(self.image_dir / row['relative_path']) as image:
            image = image.convert('RGB')
        return self.transform(image), int(row['label'])

preprocessor = ImagePreprocessor()
train_dataset = PlantVillageManifestDataset('train', preprocessor.training_transform)
val_dataset = PlantVillageManifestDataset('val', preprocessor.validation_transform)
test_dataset = PlantVillageManifestDataset('test', preprocessor.validation_transform)

loader_options = {'batch_size': BATCH_SIZE, 'num_workers': 2, 'pin_memory': device.type == 'cuda'}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print('Split sizes:', len(train_dataset), len(val_dataset), len(test_dataset))


In [ ]:
# Transfer learning: freeze all pretrained features, then train the final residual block and classifier.
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.layer4.parameters():
    parameter.requires_grad = True
model.fc = nn.Linear(model.fc.in_features, len(index_to_class))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = Adam((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE)
print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))

In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    loss_total = 0.0
    correct = 0
    sample_count = 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            loss_total += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            sample_count += labels.size(0)
    return {'loss': loss_total / sample_count, 'accuracy': correct / sample_count}

best_loss = float('inf')
epochs_without_improvement = 0
history = []
checkpoint_path = OUTPUT_DIR / 'resnet50_best.pt'

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    history.append({'epoch': epoch, 'train': train_metrics, 'validation': val_metrics})
    print(f"Epoch {epoch:02d} | train loss {train_metrics['loss']:.4f}, accuracy {train_metrics['accuracy']:.4f} | validation loss {val_metrics['loss']:.4f}, accuracy {val_metrics['accuracy']:.4f}")
    if val_metrics['loss'] < best_loss:
        best_loss = val_metrics['loss']
        epochs_without_improvement = 0
        torch.save({'model_state_dict': model.state_dict(), 'class_to_index': class_to_index, 'class_weights': class_weights.tolist()}, checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping triggered.')
            break

(OUTPUT_DIR / 'training_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

In [ ]:
def evaluate_test_set(loader):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    class_count = len(index_to_class)
    confusion = torch.zeros((class_count, class_count), dtype=torch.int64)
    with torch.no_grad():
        for images, labels in loader:
            predictions = model(images.to(device)).argmax(dim=1).cpu()
            encoded_pairs = labels * class_count + predictions
            confusion += torch.bincount(encoded_pairs, minlength=class_count ** 2).reshape(class_count, class_count)
    true_positive = confusion.diag().float()
    precision_per_class = true_positive / confusion.sum(dim=0).clamp_min(1)
    recall_per_class = true_positive / confusion.sum(dim=1).clamp_min(1)
    f1_per_class = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class).clamp_min(1e-12)
    tomato_indices = [index for index, name in enumerate(index_to_class) if name.startswith('Tomato___')]
    other_indices = [index for index, name in enumerate(index_to_class) if not name.startswith('Tomato___')]
    assert len(tomato_indices) == 10 and len(other_indices) == 28

    def group_accuracy(indices):
        support = int(confusion[indices, :].sum())
        correct = int(confusion[indices, :][:, indices].diag().sum())
        return {'accuracy': correct / support if support else 0.0, 'support': support}

    metrics = {
        'accuracy': float(true_positive.sum() / confusion.sum()),
        'precision_macro': float(precision_per_class.mean()),
        'recall_macro': float(recall_per_class.mean()),
        'f1_macro': float(f1_per_class.mean()),
        'accuracy_by_true_class_group': {
            'tomato_10_classes': group_accuracy(tomato_indices),
            'other_28_classes': group_accuracy(other_indices),
        },
        'per_class': {
            index_to_class[index]: {
                'precision': float(precision_per_class[index]),
                'recall': float(recall_per_class[index]),
                'f1': float(f1_per_class[index]),
                'support': int(confusion[index].sum()),
            } for index in range(class_count)
        },
    }
    return metrics

test_metrics = evaluate_test_set(test_loader)
(OUTPUT_DIR / 'evaluation_metrics.json').write_text(json.dumps(test_metrics, indent=2), encoding='utf-8')
print(json.dumps({key: value for key, value in test_metrics.items() if key != 'per_class'}, indent=2))
